# General Normal Inverse Example

Define a symbolic general problem with `ProblemBuilder`, generate labels with known inverse parameters, then estimate those parameters through `builder.run(...)`.


## Setup
Import the symbolic builder and create helpers for notebook-local run folders and summaries.


In [1]:
from argparse import Namespace
from pathlib import Path
import json

from kkthn.builder import ProblemBuilder


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def make_output_root(name):
    output_root = ROOT / "notebooks" / "_runs" / name
    output_root.mkdir(parents=True, exist_ok=True)
    return output_root


def latest_run_dir(output_root):
    runs = [path for path in output_root.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found under {output_root}")
    return max(runs, key=lambda path: path.stat().st_mtime)


def load_summary(run_dir):
    with open(run_dir / "summary.json", "r", encoding="utf-8") as fh:
        return json.load(fh)


def builder_args(mode, data, config, output_root):
    return Namespace(
        action="run",
        mode=mode,
        samples=data.get("num_samples"),
        epochs=config.get("epochs"),
        batch_size=config.get("batch_size"),
        seed=data.get("seed"),
        noise_scale=data.get("noise_scale"),
        output_dir=str(output_root),
    )


## Configuration
Edit these dictionaries to change the parameter box, true inverse values, initial estimates, training loop, or projection settings.


In [2]:
DATA = {
    "type": "general_normal",
    "mode": "inverse",
    "num_samples": 12,
    "seed": 42,
    "noise_scale": 0.0,
    "x_L": [-1.0, -1.0],
    "x_U": [1.0, 1.0],
    "inv_param": ["a0", "a1"],
    "inv_param_label": [1.0, -1.0],
    "inv_param_init": [10.0, -10.0],
}

CONFIG = {
    "epochs": 10,
    "batch_size": 4,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}

PROJ = {
    "fb_eps": 1e-8,
    "gn_max_iters": 25,
    "gn_tol": 1e-8,
}


## Problem Definition
The same symbolic problem can be compiled in inverse mode so the coefficients become trainable parameters.


In [3]:
def build_normal_problem(data):
    builder = ProblemBuilder(y_bound=4.0)
    x = builder.add_parameter(["x1", "x2"])
    theta = builder.add_inverse_parameter(data["inv_param"])
    y = builder.add_variable(["y1", "y2", "y3"])

    builder.objective = 0.5 * (y.y1**2 + y.y2**2 + y.y3**2)
    builder.constraints.add(
        theta.a0 * y.y1 + y.y2 - x.x1 == 0,
        y.y2 - theta.a1 * y.y3 - x.x2 == 0,
        y.y1**2 + y.y3**2 <= 2.0,
    )
    builder.bounds.set(lower=-4.0, upper=4.0)
    return builder


## Train
Create the builder instance and call `builder.run(...)`; the runner handles label generation, inverse training, and artifacts.


In [4]:
output_root = make_output_root("general_normal_inverse")
args = builder_args("inverse", DATA, CONFIG, output_root)
builder = build_normal_problem(DATA)
exit_code = builder.run(args, root=ROOT, data=DATA, train=CONFIG, proj=PROJ)
run_dir = latest_run_dir(output_root)
summary_json = load_summary(run_dir)

print(f"Exit code: {exit_code}")
print(f"Run directory: {run_dir}")
print(f"Saved artifacts: {summary_json['artifacts']}")


KKT-HardNet
  dims: n_x=4 n_y=3 n_eq=2 n_ineq=7
  mode: inverse data_n_x=2 inverse_n=2
  inverse init: {'a0': 10.0, 'a1': -10.0}
  samples: train=9 val=3 batch_size=4
  network: [2, 32, 32, 3]
epoch 0001 | train=4.198703e-01 val=3.410179e-01 raw_val=3.548660e-01 eq=1.872e-15 ineq=0.000e+00 | train_batch=1.0876s val_batch=0.0013s
epoch 0002 | train=3.990302e-01 val=3.284279e-01 raw_val=3.506750e-01 eq=1.794e-15 ineq=0.000e+00 | train_batch=0.0010s val_batch=0.0010s
epoch 0003 | train=3.814805e-01 val=3.182532e-01 raw_val=3.499074e-01 eq=1.869e-15 ineq=0.000e+00 | train_batch=0.0010s val_batch=0.0010s
epoch 0004 | train=3.658149e-01 val=3.081393e-01 raw_val=3.496773e-01 eq=1.785e-15 ineq=0.000e+00 | train_batch=0.0009s val_batch=0.0009s
epoch 0005 | train=3.517851e-01 val=2.993964e-01 raw_val=3.505203e-01 eq=1.871e-15 ineq=0.000e+00 | train_batch=0.0009s val_batch=0.0009s
epoch 0006 | train=3.398837e-01 val=2.921110e-01 raw_val=3.526771e-01 eq=1.771e-15 ineq=0.000e+00 | train_batch=0.000

## Summary
Read the emitted `summary.json`, including actual-versus-estimated inverse parameters.


In [5]:
summary = {
    "output_dir": str(run_dir),
    "dims": summary_json["dims"],
    "final": summary_json["final"],
    "component_time_percent": summary_json["timing_profile"]["component_time_percent"],
    "inverse_parameters": summary_json["inverse_parameters"]["comparison"],
}
print(json.dumps(summary, indent=2))


{
  "output_dir": "D:\\Projects\\KKTHardNet\\notebooks\\_runs\\general_normal_inverse\\builder_general_inverse_20260414_121929",
  "dims": {
    "n_eq": 2,
    "n_ineq": 7,
    "n_x": 4,
    "n_y": 3,
    "n_z": 19,
    "raw_n_ineq": 1
  },
  "final": {
    "epoch": 10,
    "mean_batch_loss": 0.5521095689823657,
    "train_batches": 3,
    "train_epoch_time_sec": 0.00259150000000119,
    "train_eq_l2": 3.5847379318857944e-15,
    "train_eval_time_sec": 0.0004989000000001909,
    "train_ineq_l2": 0.0,
    "train_loss": 0.3084064758945882,
    "train_raw_mse": 0.7964206453944582,
    "train_step_time_per_batch_sec": 0.00044263333333323845,
    "train_step_time_sec": 0.0013278999999997154,
    "train_time_per_batch_sec": 0.00086383333333373,
    "val_eq_l2": 1.7727935198186339e-15,
    "val_ineq_l2": 0.0,
    "val_loss": 0.27169459782030303,
    "val_raw_mse": 0.36768848156590894,
    "validation_batches": 1,
    "validation_epoch_time_sec": 0.0010272000000028925,
    "validation_time_per